In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

In [ ]:
#======= Read in Performance Metrics Files =======
summary = pd.read_csv('../../../../CO2-CH4/irmof1/Data_Size_Test/Summary_metrics.csv')
# actual initial size (n_iters counts the first fit): PX 35 -> 36, PXT 15 -> 20 and 30 -> 40
summary['n_init'] = (summary.n_train_final_mean - summary.n_iters_mean + 1).round().astype(int)
# PXT 15 = 20 and 30 = 40 are the same run: keep one
summary = summary.drop_duplicates(['run', 'space', 'n_init']).sort_values('n_init')
# standard error of the mean over the 5 data splits
summary['MAPE_se'] = summary.MAPE_std / np.sqrt(summary.n_seeds)
summary['n_iters_mean'] = summary.n_iters_mean.round().astype(int)

def metrics(run, space):
    # same column order as the old Metrics files (size, R2, MAE, MAPE, iterations) + MAPE standard error
    d = summary[(summary.run == run) & (summary.space == space)]
    return d[['n_init', 'R2_mean', 'MAE_mean', 'MAPE_mean', 'n_iters_mean', 'MAPE_se']].reset_index(drop=True)

In [ ]:
#======= Define Plot Values and Generate Plots ========
# P-X Dataframes
ch_px = metrics('TAL_Pure_CH_CH_M', 'px')
print(ch_px)
co_px = metrics('TAL_Pure_CO_CO_M', 'px')

# P-X Values
data_size_px_ch = ch_px.iloc[:, 0]
data_size_px_co = co_px.iloc[:, 0]

# CO2 metrics
mape_co2_px = co_px.iloc[:, 3]
no_iters_co2_px = co_px.iloc[:, 4]

# CH4 metrics
mape_ch4_px = ch_px.iloc[:, 3]
no_iters_ch4_px = ch_px.iloc[:, 4]

# Standard Error Values
se_co2_px = co_px.iloc[:, 5]
se_ch4_px = ch_px.iloc[:, 5]

# Generate Plot
panel_labels = ["(a)", "(b)"]
fig, ax = plt.subplots(figsize=(8, 6))

scale = 2  # adjust bubble size

# P-X Plot
for i, ds in enumerate(data_size_px_co):
    ax.scatter(ds, mape_co2_px[i]*100, s=no_iters_co2_px[i] * scale, 
               color='steelblue', alpha=0.5, zorder=5, edgecolors = 'darkblue')
for i, ds in enumerate(data_size_px_ch):
    ax.scatter(ds, mape_ch4_px[i]*100, s=no_iters_ch4_px[i] * scale, 
               color='darkgreen', alpha=0.5, zorder=5, edgecolors = 'black')

# Connect bubbles with a thin line to show trend
ax.plot(data_size_px_co, mape_co2_px*100, color='steelblue',  linestyle='--', linewidth=2.0, label=r'$\mathrm{CO}_{2}$')
ax.plot(data_size_px_ch, mape_ch4_px*100, color='darkgreen', linestyle='--', linewidth=2.0, label=r'$\mathrm{CH}_{4}$')

ax.errorbar(data_size_px_co, mape_co2_px*100, yerr=se_co2_px*100, fmt='none', ecolor='darkblue',
               elinewidth=1.5, capsize=4, capthick=1.5, zorder=10)
ax.errorbar(data_size_px_ch, mape_ch4_px*100, yerr=se_ch4_px*100, fmt='none', ecolor='black',
               elinewidth=1.5, capsize=4, capthick=1.5, zorder=10)
ax.margins(x=0.06, y=0.10)

# Size legend
for iters in [min(no_iters_ch4_px), max(no_iters_co2_px)]:
    ax.scatter([], [], s=iters * scale, color='grey', alpha=0.5,
               label=f'{iters} TAL iters')

ax.set_xlabel('Initial Training Data Size', fontsize = 15, fontweight="bold")
ax.set_ylabel('MAPE (%)', fontsize = 15, fontweight="bold")
for tick in ax.get_xticklabels() + ax.get_yticklabels():
    tick.set_fontsize(13)
    tick.set_fontweight("bold")
# ax.set_title('TAL Performance vs Starting Mixture Data Size - P-X Space', fontsize = 15)
ax.legend(loc='upper left')
ax.grid(True, linestyle='--', alpha=0.4)
plt.tight_layout()
plt.savefig('Data_size_test.png', dpi=600, bbox_inches="tight", pad_inches=0.05, facecolor="white")
plt.savefig('Data_size_test.pdf', dpi=600, bbox_inches="tight", pad_inches=0.05, facecolor="white")
plt.show()